# 02 Wasserstein Regime Clustering

This notebook reproduces the core idea: cluster rolling windows of SPY returns as empirical distributions using Wasserstein k-means.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
from regime_ot.baselines import extract_moment_features, run_standard_kmeans
from regime_ot.data import download_prices
from regime_ot.evaluation import silhouette_from_distance_matrix
from regime_ot.plotting import plot_centroid_distributions, plot_price_with_regimes
from regime_ot.returns import clean_returns, compute_log_returns
from regime_ot.wasserstein import pairwise_wasserstein_distance
from regime_ot.windows import make_rolling_windows
from regime_ot.wkmeans import WassersteinKMeans

## Download SPY and build return windows

In [ ]:
prices = download_prices(['SPY'], start='2000-01-01')
spy = prices['SPY']
returns = clean_returns(compute_log_returns(spy)).dropna()
windows, dates = make_rolling_windows(returns, window_size=63, step=5)
windows.shape

## Fit Wasserstein k-means for k=2, k=3, and k=4

In [ ]:
distance_matrix = pairwise_wasserstein_distance(windows)
models = {}
scores = []

for k in [2, 3, 4]:
    model = WassersteinKMeans(n_clusters=k, random_state=42, n_init=10).fit(windows)
    score = silhouette_from_distance_matrix(distance_matrix, model.labels_)
    models[k] = model
    scores.append({'k': k, 'inertia': model.inertia_, 'silhouette': score})

pd.DataFrame(scores)

Use silhouette together with interpretability. For a first pass, k=3 often gives a readable low/medium/high risk regime split.

In [ ]:
selected_k = 3
model = models[selected_k]

fig, ax = plot_price_with_regimes(spy, model.labels_, dates)
fig.savefig(FIGURES / 'SPY_regimes.png', dpi=150)

fig, ax = plot_centroid_distributions(model.centroids_)
fig.savefig(FIGURES / 'SPY_centroids.png', dpi=150)

## Compare against KMeans on moment features

In [ ]:
features = extract_moment_features(windows)
baseline = run_standard_kmeans(features, n_clusters=selected_k)
features.assign(wasserstein_label=model.labels_, moment_kmeans_label=baseline['labels']).head()